# ** 1. Install Required Libraries**

In [ ]:
# Keep the numpy/scipy versions already loaded in Colab, so pip does not replace them
import numpy, scipy
with open("/content/constraints.txt", "w") as f:
    f.write(f"numpy=={numpy.__version__}\nscipy=={scipy.__version__}\n")

!apt-get -qq install ffmpeg
!pip -q install -c /content/constraints.txt openai-whisper pyannote.audio jiwer
!pip -q install -c /content/constraints.txt -U google-genai
!pip -q install pyannote.audio jiwer
!apt-get -qq install ffmpeg
!pip -q install openai-whisper
!pip -q install -U google-genai
!pip install -U pyannote.audio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 45.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.


# 2. Upload the Audio File

In [ ]:
from google.colab import files

uploaded = files.upload()
audio_path = next(iter(uploaded))

print("Uploaded file:", audio_path)

Saving WhatsApp Audio 2026-10-20 at 12.57.04 AM (1).mp4 to WhatsApp Audio 2026-10-20 at 12.57.04 AM (1).mp4
Uploaded file: WhatsApp Audio 2026-10-20 at 12.57.04 AM (1).mp4


# 3. Transcribe the Audio with Whisper

In [ ]:

import time
import whisper

model = whisper.load_model("large-v3")


audio = whisper.load_audio(audio_path)
audio_duration = len(audio) / 16000  # Duration in seconds

# condition_on_previous_text=False: This prevents repeated sentences and stops one mistake from spreading to the rest of the audio.
start = time.time()
result = model.transcribe(
    audio,
    condition_on_previous_text=False,
    word_timestamps=True,
    hallucination_silence_threshold=2
)
whisper_time = time.time() - start  # Used later to compute RTF

print("Detected language:", result["language"])
print(result["text"])

100%|██████████████████████████████████████| 2.88G/2.88G [00:29<00:00, 106MiB/s]


Detected language: en
 So you're going to clear her however you do this with Evelyn, right? Because she's the one that has thoughts about the language and stuff. Yes. But in our discussion, we haven't mentioned, it was assumed there'd be a voiceover. Well, remind her that there was a question about when they're on the one.


# 4. Speaker Diarization with PyAnnote

In [ ]:
# Run PyAnnote to find who spoke when
import time
import torch
from google.colab import userdata
from pyannote.audio import Pipeline

# Load the diarization model
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=userdata.get("HF_TOKEN")
)
pipeline.to(torch.device("cuda"))

# Give the audio to the model
waveform = torch.from_numpy(audio).unsqueeze(0)

start = time.time()  # Start the stopwatch
output = pipeline({"waveform": waveform, "sample_rate": 16000})
diarization_time = time.time() - start  # Stop the stopwatch (seconds)

# Keep the version where only one speaker talks at a time
exclusive_diarization = output.exclusive_speaker_diarization

print("Speakers found:", len(output.speaker_diarization.labels()))

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/xvec_transform.npz: downloading bytes:           |  0.00B            

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

... (2 outputs left out)

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


Speakers found: 2


/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)


# 5. Build the Speaker-Attributed Transcript

In [ ]:
# Build a transcript with timestamps and speaker labels
from pyannote.core import Segment

turns = []  # Each turn: speaker, start, end, text
speaker = "Unknown"
current_speaker = None

for segment in result["segments"]:
    for i, word in enumerate(segment["words"]):

        # Find who is speaking while this word is said
        label = exclusive_diarization.crop(Segment(word["start"], word["end"])).argmax()
        if label:  # If nobody is found, keep the previous speaker
            speaker = label.replace("SPEAKER_", "Speaker ")

        # Same speaker in the same segment: continue the turn, otherwise start a new one
        if i > 0 and speaker == current_speaker:
            turns[-1]["text"] += word["word"]
            turns[-1]["end"] = word["end"]
        else:
            turns.append({"speaker": speaker, "start": word["start"], "end": word["end"], "text": word["word"]})
            current_speaker = speaker

# Format each turn like: [00:12-00:18] Speaker 01: text
transcript_lines = []
for turn in turns:
    start = int(turn["start"])
    end = int(turn["end"])
    start_time = f"{start // 60:02d}:{start % 60:02d}"
    end_time = f"{end // 60:02d}:{end % 60:02d}"
    transcript_lines.append(f"[{start_time}-{end_time}] {turn['speaker']}: {turn['text'].strip()}")

transcript = "\n".join(transcript_lines)
print(transcript)

[00:00-00:02] Speaker 01: So you're going to clear her however you do this with Evelyn, right?
[00:02-00:04] Speaker 01: Because she's the one that has thoughts about the language
[00:04-00:05] Speaker 00: and stuff.
[00:05-00:05] Speaker 00: Yes.
[00:06-00:08] Speaker 00: But in our discussion, we haven't mentioned,
[00:08-00:09] Speaker 00: it was assumed there'd be a voiceover.
[00:10-00:10] Speaker 01: Well,
[00:10-00:12] Speaker 00: remind her that there was a question about
[00:12-00:13] Speaker 01: when they're
[00:13-00:13] Speaker 00: on the one.


# 6. Measure Speed: Real-Time Factor (RTF)
This step measures how long the system takes to process the audio clip.


In [ ]:
# Compute RTF for Whisper + PyAnnote (model loading and Gemini time are not included)
rtf = (whisper_time + diarization_time) / audio_duration

print(f"Audio duration: {audio_duration:.1f} s")
print(f"Whisper: {whisper_time:.1f} s | PyAnnote: {diarization_time:.1f} s")
print(f"RTF: {rtf:.2f}")

Audio duration: 13.7 s
Whisper: 7.4 s | PyAnnote: 1.1 s
RTF: 0.62


### 6. Set Up the Gemini Client

This step connects the notebook to Gemini, which writes the meeting summary. It needs your own Gemini API key.

**How to get your API key:**
1. Go to Google AI Studio: https://aistudio.google.com/apikey
2. Sign in with your Google account.
3. Click **Create API key**.
4. Copy the key.

**How to add it to Colab:**
1. Click the key icon 🔑 in the left sidebar of Colab.
2. Click **Add new secret**.
3. Set the name to `GEMINI_API_KEY` (it must match exactly).
4. Paste your key in the **Value** field.
5. Turn on **Notebook access**.

> Never paste your key directly in the code and never share it with others.





In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

### 7. Correct Transcription Errors with Gemini

In [ ]:

from google.genai import types

correction_prompt = f"""
You are a careful proofreader for a meeting transcript (Arabic and/or English) produced by speech recognition.

Rules:
- Fix only obvious recognition errors using the context of the conversation.
- Keep every timestamp and every speaker label (like "Speaker 01") exactly as it is.
- Keep each line in its original language (do not translate).
- Do not add, remove, merge, or invent any information.
- If a word or sentence is unclear and you are not sure, keep it as it is and add [?] after it.

Transcript:
{transcript}
"""

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=correction_prompt,
    config=types.GenerateContentConfig(temperature=0)  # Less guessing, more stable output
)

corrected_transcript = response.text
print(corrected_transcript)

### 8. Build the Meeting Prompt

In [ ]:
prompt = f"""
You are an assistant specialized in meeting minutes.
Write the output in the same language as the transcript (Arabic or English).

Use only the meeting transcript below. Each line has a timestamp and a speaker label.
Do not invent names, decisions, deadlines, or action items.
Speaker labels are anonymous (like Speaker 00, Speaker 01). Use a real name only if it is said in the conversation.
If something is unclear or not finalized, label it "Uncertain".
Cite every point and every action item with its speaker and start time, like [Speaker 01 | 04:12].

Organize the result under these headings:
1. Meeting summary
2. Key discussion points
3. Confirmed decisions
4. Action items: assignee, task, deadline, citation
5. Open or uncertain items

Transcript:
{corrected_transcript}
"""

### 9. Generate the Meeting  with Gemini

In [ ]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=prompt
)

summary  = response.text
print(summary)

In [ ]:
# Save the summary and the transcript in one file and download it
from google.colab import files

with open("meeting_report.txt", "w", encoding="utf-8") as file:
    file.write("===== Summary =====\n\n")
    file.write(summary)
    file.write("\n\n===== Transcript =====\n\n")
    file.write(transcript)

files.download("meeting_report.txt")